# Reproducing the VCell hybrid methods paper (Schaff et al. 2016) with the native solver and the co-simulation

_Investigation `vcell-hybrid-paper-benchmarks` — coder reproduction notebook._

**Question.** Do VCell's native PDE/particle hybrid solver (vcell-fvsolver with embedded Smoldyn) and the process-bigraph co-simulation both reproduce the validation results of Schaff et al. (2016, PLoS Comput Biol 12:e1005236), and where do they differ?

A reproduction of the published validation of VCell's hybrid solver, used both to validate the headless native reference path and to benchmark the co-simulation on established cases.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-pde-particle/viva-pde-particle').is_dir():
    REPO = Path('/home/runner/work/viva-pde-particle/viva-pde-particle')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_pde_particle.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Schaff et al. 2016, Test 1 / Fig 1B: separable calcium sparks on both solvers (`separable-calcium-sparks`)

**Question.** Do the native VCell hybrid solver and the process-bigraph co-simulation both reproduce
Fig 1B of Schaff et al. (2016)? The error of the trial-mean calcium against its exact
expectation should fall as N^-1/2 with the number of trials, and be statistically
dominated at small Δt.

**Objective.** Model (Eq. 1 of the paper): ∂U/∂t = D∇²U + J·Σᵢ ξᵢ δ(r − rᵢ) − V_p(U − U₀), no-flux
boundaries. D = 1 µm²/s, J = 10 µM·µm³/s, U₀ = 0.1 µM, V_p = 1 s⁻¹, k_on = 1 s⁻¹,
k_off = 5 s⁻¹, 24 channels starting closed, T = 5 s.
Geometry: [0, 10.1]×[0, 2.1]×[0, 0.5] µm³, Δx = Δy = 0.1.
Runs: 128 trials per solver at Δt ∈ {0.02, 0.01, 0.005} s. ε(N) is taken from nested
subsets at Δt = 0.005, and ε is the RMS over space and time of (trial mean − E[U]),
relative to max(E[U] − U₀).

**Hypothesis.** The channels' gating is independent of calcium, so E[U] solves the linear PDE driven by
the exact open probability p(t) = k_on/(k_on+k_off)·(1 − e^-(k_on+k_off)t). Both solvers'
trial means converge to it with ε ∝ N^-1/2. At N = 128 the statistical error dominates
the Δt error for Δt ≤ 0.02 s. The two solvers differ only by sampling.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `calcium-sparks-test1` | `viva_pde_particle.composites.examples.calcium_sparks` | 0 | test=1, dt=0.005 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_pde_particle.composites.examples.calcium_sparks`** — `spec_viva_pde_particle_composites_examples_calcium_sparks` (a plain, editable dict)


_composite spec file for `viva_pde_particle.composites.examples.calcium_sparks` not found under `viva_pde_particle/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: separable-calcium-sparks ===
STUDY = 'separable-calcium-sparks'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**calcium-sparks-fig1b**


In [ ]:
# calcium-sparks-fig1b
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**sparks-3d**


In [ ]:
# sparks-3d
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**sparks-3d-movie**


In [ ]:
# sparks-3d-movie
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**sparks-3d-interactive**


In [ ]:
# sparks-3d-interactive
show_viz(_render_one('threejs:viz/sparks_3d_interactive.html', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| cosim-error-falls-as-inverse-sqrt-n | kind=derived_scalar field=c1_cosim_eps_slope | op range low -0.75 high -0.25 provenance {'kind': 'literature', 'note': 'Schaff et al. 2016, Fig 1B; Monte Carlo error ∝ N^-1/2 when statistical error dominates.'} |
| native-error-falls-as-inverse-sqrt-n | kind=derived_scalar field=c1_native_eps_slope | op range low -0.75 high -0.25 provenance {'kind': 'literature', 'note': 'Schaff et al. 2016, Fig 1B.'} |
| cosim-error-small-at-128-trials | kind=derived_scalar field=c1_cosim_eps_n128 | op < value 0.03 provenance {'kind': 'theory', 'note': 'About 4% at N = 16 scales to about 1.4% at N = 128 under N^-1/2.'} |
| solvers-differ-only-by-sampling | kind=derived_scalar field=c1_cosim_vs_native_eps_ratio | op < value 1.5 provenance {'kind': 'theory', 'note': 'Independent ensembles of the same expectation; the ratio is about 1 when only sampling differs.'} |


## Study: Schaff et al. 2016, Test 2 / Fig 2: coupled calcium sparks in the fast-diffusion limit, both solvers (`coupled-sparks-fast-diffusion`)

**Question.** With channel opening coupled to local calcium (k_on·U(rᵢ)/U₀) and fast diffusion
(D = 1000 µm²/s), do both hybrid solvers reproduce the distributions of calcium and
open-channel number given by the exact well-mixed process, as in Fig 2 of the paper?

**Objective.** Parameters as in Test 1, except k_on = 0.1 s⁻¹ (opening at k_on·U/U₀), D = 1000 µm²/s
and Δt = 0.2 ms; 500 trials per solver.
The reference is 20,000 exact realizations of the well-mixed process with
V = 10.1×2.1×0.5 µm³.
At t = 1, 2, 3 s the comparison uses:
- mean |z| of U and n against the reference;
- the L2 distance between histograms, as in the paper;
- two-sample KS tests, against the reference and between the solvers.

**Hypothesis.** At D = 1000 the cell is effectively well mixed. The spatial hybrid's volume-averaged
U and open count n then follow the well-mixed piecewise-deterministic process, which
the paper sampled with the Gibson–Bruck method; here it is sampled exactly by
event-driven simulation (viva_pde_particle.benchmarks.pdmp).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `calcium-sparks-test2` | `viva_pde_particle.composites.examples.calcium_sparks` | 0 | test=2, dt=0.0002 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_pde_particle.composites.examples.calcium_sparks`** — `spec_viva_pde_particle_composites_examples_calcium_sparks` (a plain, editable dict)


_composite spec file for `viva_pde_particle.composites.examples.calcium_sparks` not found under `viva_pde_particle/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: coupled-sparks-fast-diffusion ===
STUDY = 'coupled-sparks-fast-diffusion'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**calcium-sparks-fig2**


In [ ]:
# calcium-sparks-fig2
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**coupled-sparks-3d**


In [ ]:
# coupled-sparks-3d
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**coupled-sparks-3d-movie**


In [ ]:
# coupled-sparks-3d-movie
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**coupled-sparks-3d-interactive**


In [ ]:
# coupled-sparks-3d-interactive
show_viz(_render_one('threejs:viz/coupled_sparks_3d_interactive.html', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| cosim-matches-well-mixed-means | kind=derived_scalar field=c2_cosim_max_mean_z | op < value 4.0 provenance {'kind': 'theory', 'note': 'Fast-diffusion limit; 500 trials against 20,000 exact realizations.'} |
| native-matches-well-mixed-means | kind=derived_scalar field=c2_native_max_mean_z | op < value 4.0 provenance {'kind': 'theory', 'note': 'As above.'} |
| cosim-channel-distribution-l2 | kind=derived_scalar field=c2_cosim_max_l2_n | op < value 0.1 provenance {'kind': 'literature', 'note': "The paper's metric (Fig 2, 10,000 realizations gave about 0.002–0.003). With 500 trials the sampling floor is about 0.05."} |
| native-channel-distribution-l2 | kind=derived_scalar field=c2_native_max_l2_n | op < value 0.1 provenance {'kind': 'literature', 'note': 'As above.'} |
| solvers-agree | kind=derived_scalar field=c2_cosim_vs_native_min_ks_pvalue | op > value 0.001 provenance {'kind': 'theory', 'note': 'The two solvers should agree; 6 comparisons with ties in n make KS conservative.'} |


## Study: Schaff et al. 2016, Tests 3–5 / Figs 4–6: a coupled single channel against direct Fokker–Planck solutions (`single-channel-fokker-planck`)

**Question.** With a single channel whose opening is coupled to calcium (rate αβ(ρ(0) + 1)), does the hybrid
co-simulation reproduce the probability densities of calcium given by direct, Monte-Carlo-free
solutions of the Fokker–Planck equations? Two regimes: fast diffusion (well mixed, Test 3) and
finite diffusion on coarse grids (Tests 4–5), as in Figs 4–6 of the paper.

**Objective.** Dimensionless model: ∂τρ = d·∂²ₓρ + a·δ(x)·ξ − ρ. The channel opens at αβ(ρ(0) + 1) and closes
at α; it starts closed, with ρ = 0.

- **Test 3:** α = β = 1, a = 24, d = 10⁴, 3D cube of side 1 (Δx = 0.1, channel at the centre),
  Δτ = 0.002.
  - Compared: p(ρ̄) at τ = 30.
  - Reference: Eq 4, solved in time and checked against its exact steady state.
  - Ensembles: 10,000 co-sim trials, and the same on native VCell.
- **Test 4:** α = 10, β = 1, a = 20/3, d = 1, two cells with Δx = 2, Δτ = 10⁻⁴.
  - Compared: p(ρᵢ) at τ = 1.
  - Reference: the S2-Text master equation (Eq T2.1) at Δρ = 2.5, 1.2 and 0.6 × 10⁻³, extrapolated
    to Δρ → 0.
  - Ensembles: 12,500 trials.
- **Test 5:** α = 20, β = 0.5, a = 20, d = 50, three cells.
  - Reference: four Δρ, extrapolated as in the paper.
  - Ensembles: 12,500 trials at Δτ = 10⁻⁴.

Densities are 20-bin histograms on [0, ρ_max(i)] (40 bins on [0, 24] for Test 3), compared with the
reference averaged over the same bins. Reported: the relative L2 (RMS over bins / reference
maximum), the absolute L2, and the reduced χ² against multinomial sampling noise.

**Hypothesis.** The hybrid samples the same piecewise-deterministic process the Fokker–Planck equations
describe, so its histograms should match them to within sampling noise (reduced χ² ≈ 1). The
paper reports L2 differences of 1–3% of the maximum.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `single-channel` | `viva_pde_particle.composites.examples.calcium_sparks` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_pde_particle.composites.examples.calcium_sparks`** — `spec_viva_pde_particle_composites_examples_calcium_sparks` (a plain, editable dict)


_composite spec file for `viva_pde_particle.composites.examples.calcium_sparks` not found under `viva_pde_particle/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: single-channel-fokker-planck ===
STUDY = 'single-channel-fokker-planck'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fokker-planck**


In [ ]:
# fokker-planck
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| test3-cosim-matches-fokker-planck | kind=derived_scalar field=c3_test3_cosim_chi2 | op range low 0.0 high 2.0 provenance {'kind': 'theory', 'note': 'Sampling noise alone gives ≈ 1 (the 99th percentile of χ²/k for k ≈ 20–40 bins is about 1.7–1.9).'} |
| test3-native-matches-fokker-planck | kind=derived_scalar field=c3_test3_native_chi2 | op range low 0.0 high 2.0 provenance {'kind': 'theory', 'note': 'As above.'} |
| test4-cosim-matches-fokker-planck | kind=derived_scalar field=c3_test4_i0_cosim_chi2 | op range low 0.0 high 2.0 provenance {'kind': 'theory', 'note': 'As above. The paper reports a relative L2 of 1.9% (i = 0) and 3.3% (i = 1).'} |
| test5-cosim-matches-fokker-planck | kind=derived_scalar field=c3_test5_i0_cosim_chi2 | op range low 0.0 high 2.0 provenance {'kind': 'theory', 'note': 'As above. The paper reports 1.3–1.5%.'} |
